# VoxEntity-EU — CPU reproduction notebook (offline)

From the release bundle's frozen predictions: verify hashes and regenerate every table without internet access. This is reproduction level 1 (identical scoring from frozen predictions).

In [ ]:
# 1. verify bundle checksums
import hashlib
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
bundle = root / 'release_bundle'
bad = []
for line in (bundle / 'checksums.sha256').read_text().splitlines():
    digest, rel = line.split()
    if hashlib.sha256((bundle / rel).read_bytes()).hexdigest() != digest:
        bad.append(rel)
assert not bad, f'checksum mismatch: {bad}'
print('checksums ok')


In [ ]:
# 2. rescore frozen predictions (CPU, deterministic, no network) and compare
import json

from voxentity_eu.cli import main as cli_main

out = root / 'reproduced'
out.mkdir(exist_ok=True)
runs = ['gold-extractor', 'gold-written-extractor', 'whisper-turbo-slots', 'voxtral-mini-slots', 'reference-agent']
for run in runs:
    pred = bundle / 'predictions' / run / 'predictions.jsonl'
    assert pred.exists(), f'missing frozen predictions: {pred}'  # never skip silently
    rc = cli_main(['score', '--gold', str(bundle / 'gold/scenarios.jsonl'),
                   '--predictions', str(pred),
                   '--manifest', str(bundle / 'audio_manifest.jsonl'),
                   '--out', str(out / f'{run}.json')])
    assert rc == 0, (run, rc)
    mine = json.loads((out / f'{run}.json').read_text(encoding='utf-8'))['summary']['overall']
    ref = json.loads((bundle / 'predictions/scored' / f'{run}.json').read_text(encoding='utf-8'))['summary']['overall']
    assert mine == ref, f'{run}: reproduced summary differs from bundle'
    print(run, 'slot accuracy', round(100 * mine['slot_accuracy'], 1), 'identical to bundle')


In [ ]:
# 3. regenerate the report tables from the reproduced scores
print(cli_main(['report', '--results', str(out), '--out', str(out / 'report.md')]))
